In [171]:
%load_ext autoreload
%autoreload 2 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# AISDI - ćwiczenie 2
## [1] Wprowadzenie
Celem drugiego zadania z przedmiotu Algorytmy i Struktury Danych (AISDI) była implementacja oraz analiza wydajności algorytmów grafowych na przykładzie miejskiej sieci drogowej. Głównym zadaniem było zaimportowanie danych zawierających odległości oraz czasy przejazdu między węzłami w różnych porach dnia, a następnie wyznaczenie optymalnych tras na podstawie kryterium dystansu oraz czasu.

### [1.1] Zaimportowane biblioteki
Zgodnie z wymaganiami projektowymi, do reprezentacji grafu nie wykorzystano gotowych bibliotek (takich jak networkx). Graf został zaimplementowany od podstaw przy użyciu zmodyfikowanej listy sąsiedztwa.

Główną strukturą przechowującą dane jest słownik, którego kluczem jest identyfikator węzła (miasta), a wartością lista obiektów klasy Destination.
Zastosowane narzędzia i moduły:

- csv – do wczytywania i parsowania plików z danymi.

- destination.py – autorska klasa reprezentująca krawędź grafu (cel podróży) wraz z jej wagami (dystans, czas w zależności od pory dnia).

- graph_logic.py – moduł zawierający implementację algorytmów trasowania.

- time – do przeprowadzania pomiarów wydajności i czasu kompilacji algorytmów.

In [180]:
import csv
from pprint import pprint
from destination import Destination
from graph_logic import *
import time

### [1.2] Analiza danych wejściowych
Dane wejściowe pochodzą z pliku city_small.csv. Każdy wiersz reprezentuje krawędź grafu nieskierowanego (dwukierunkową drogę) wraz z zestawem wag. Struktura danych prezentuje się następująco:

In [181]:
with open("data/city_small.csv", newline="") as csv_file:
    small_cities = csv.reader(csv_file)
    for index, row in enumerate(small_cities):
        if index > 5:
            break
        print(row)

['od', 'do', 'odleglosc', 'czas_rano', 'czas_poludnie', 'czas_wieczor']
['0', '9', '17.1', '19', '11', '14']
['0', '20', '6.84', '11', '5', '8']
['1', '11', '9.4', '22', '12', '12']
['1', '21', '8.12', '13', '11', '15']
['1', '22', '19.55', '33', '20', '20']


Konstrukcja grafu odbywa się poprzez iterację po pliku CSV i dwukrotne dodanie krawędzi (od A do B oraz od B do A) do słownika pathways za pomocą metody create_destination().


## [2] Wyznaczanie najkrótszej ścieżki
W pierwszej fazie badań przetestowano działanie klasycznego algorytmu Dijkstry, szukającego najkrótszej ścieżki od wybranego wierzchołka do pozostałych węzłów na podstawie wag odległościowych. Zbadano zachowanie algorytmu w zależności od rozmiaru grafu.

### [2.1] Małe miasto
Algorytm dla małej siatki skrzyżowań odnalazł trasy błyskawicznie. W wynikach prawidłowo zidentyfikowano braki połączeń między odizolowanymi węzłami (oznaczone wartością inf), co świadczy o tym, że badany podgraf nie jest w pełni spójny.

In [182]:

small_pathways = import_city("data/city_small.csv")

start_time = time.perf_counter()
cities_info = get_dijkstra_info_distance(49, small_pathways)
end_time = time.perf_counter()
total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for small city dijkstra is {total_time:.04f}")


Distance from 8 to 40 is 67.47
Distance from 2 to 39 is 76.97
Distance from 34 to 34 is 0.00
Distance from 33 to 0 is inf
Distance from 44 to 49 is 132.37
Distance from 10 to 39 is 84.23
Distance from 2 to 12 is 41.91
Distance from 30 to 31 is 44.50
Distance from 44 to 48 is 108.82
Distance from 9 to 47 is inf

Total time for small city dijkstra is 0.0004


### [2.2] Średnie miasto
Dla średniego zbioru danych zauważono oczekiwany wzrost czasu wykonywania algorytmu, związany z większą liczbą krawędzi poddawanych procesowi relaksacji.

In [183]:
mid_pathways = import_city("data/city_mid.csv")

start_time = time.perf_counter()
cities_info = get_dijkstra_info_distance(199, mid_pathways)
end_time = time.perf_counter()

total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for mid city dijkstra is {total_time:.04f}")

Distance from 50 to 152 is 43.06
Distance from 4 to 51 is 12.56
Distance from 106 to 168 is 23.93
Distance from 77 to 83 is 71.74
Distance from 113 to 139 is 44.75
Distance from 93 to 129 is 22.81
Distance from 132 to 106 is 45.03
Distance from 97 to 103 is 18.95
Distance from 118 to 68 is 52.28
Distance from 45 to 17 is 38.74

Total time for mid city dijkstra is 0.0026


### [2.3] Duże miasto 

In [184]:
big_pathways = import_city("data/city_big.csv")

start_time = time.perf_counter()
cities_info = get_dijkstra_info_distance(999, big_pathways)
end_time = time.perf_counter()

total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for big city dijkstra is {total_time:.04f}")

Distance from 187 to 655 is 37.56
Distance from 100 to 218 is 81.33
Distance from 752 to 60 is 66.70
Distance from 174 to 839 is 62.83
Distance from 367 to 589 is 46.86
Distance from 73 to 365 is 39.88
Distance from 923 to 201 is 52.95
Distance from 568 to 134 is 51.30
Distance from 861 to 191 is 54.48
Distance from 668 to 366 is 90.45

Total time for big city dijkstra is 0.0244


### [2.4] Porównanie wydajności
 Czas wyznaczania dystansu skaluje się niezwykle dobrze. Dzięki zastosowaniu kopca dla kolejki priorytetowej, algorytm unika niepotrzebnych iteracji, co pozwala rozwiązywać najkrótszą ścieżkę dla dużych sieci drogowych w zadowalającym, ułamkowym czasie.

## Czas
W tym badaniu zastosowano zmodyfikowaną wersję algorytmu Dijkstry. Jej specyfika polega na śledzeniu upływającego czasu (w minutach od północy) i dynamicznym dobieraniu odpowiedniej wagi (czas rano, w południe lub wieczorem) podczas wjazdu na kolejne skrzyżowanie. Czas początkowy jest ustawiony na 7:30

### [3.1] Małe miasto


In [185]:
start_time = time.perf_counter()
cities_info = get_dijkstra_info_time(49, small_pathways, "07:30")
end_time = time.perf_counter()
total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for small city dijkstra time is {total_time:.04f}")
print(f"This simulation is for starting time equal 07:30")

Arrival time from 9 to 41 is inf
Arrival time from 42 to 12 is 09:33
Arrival time from 13 to 46 is 09:52
Arrival time from 43 to 21 is 10:08
Arrival time from 0 to 42 is inf
Arrival time from 31 to 47 is 10:13
Arrival time from 6 to 48 is 10:22
Arrival time from 22 to 3 is inf
Arrival time from 47 to 36 is 10:01
Arrival time from 13 to 37 is 09:29

Total time for small city dijkstra time is 0.0006
This simulation is for starting time equal 07:30


### [3.2] Średnie miasto


In [186]:
start_time = time.perf_counter()
cities_info = get_dijkstra_info_time(199, mid_pathways, "07:30")
end_time = time.perf_counter()
total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for  calculating medium city dijkstra time is {total_time:.04f}")
print(f"This simulation is for starting time equal 07:30")

Arrival time from 132 to 30 is 09:27
Arrival time from 17 to 107 is 08:56
Arrival time from 176 to 181 is 08:00
Arrival time from 133 to 61 is 09:17
Arrival time from 70 to 102 is 08:00
Arrival time from 55 to 171 is 08:11
Arrival time from 161 to 3 is 09:25
Arrival time from 80 to 90 is 08:48
Arrival time from 97 to 72 is 09:02
Arrival time from 142 to 93 is 07:46

Total time for  calculating medium city dijkstra time is 0.0048
This simulation is for starting time equal 07:30


### [3.3] Duże miasto

In [188]:
start_time = time.perf_counter()
cities_info = get_dijkstra_info_time(999, big_pathways, "07:30")
end_time = time.perf_counter()
total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for  calculating big city dijkstra time is {total_time:.04f}")
print(f"This simulation is for starting time equal 07:30")

Arrival time from 6 to 537 is 08:30
Arrival time from 200 to 228 is 07:43
Arrival time from 36 to 965 is 08:34
Arrival time from 363 to 794 is 08:43
Arrival time from 403 to 84 is 09:02
Arrival time from 292 to 556 is 08:12
Arrival time from 645 to 71 is 08:52
Arrival time from 498 to 750 is 09:01
Arrival time from 931 to 444 is 08:11
Arrival time from 240 to 684 is 09:04

Total time for  calculating big city dijkstra time is 0.0775
This simulation is for starting time equal 07:30


### [3.4] Wnioski
Zastosowanie zmodyfikowanej wersji algorytmu Dijkstry
do obliczania czasu przejazdu kończy się pełnym sukcesem.
Czas wykonania (0.06 s) udowadnia, że dynamiczne
pobieranie wag (rano/południe/wieczór) w zależności od
bieżącej godziny nie spowalnia w zauważalny sposób
działania programu dla małych struktur.

## Komiwojazer:
Ten etap był najtrudniejszy do zrealizowania, po pierwsze w wielu przypadkach dotyczących grafu nie mam możliwości przejscia z jednegoe miasta do drugiego bezposrednio, dlatego trzeba zawracać. Dodatkowo niektóre miasta  tworzą graf, który jest odizolowany od głownego grafu, co powoduje, że klaysyczne rozwiązanie problemu kowimojażera staje się niemożliwe, a do tego należy wspomnieć, że tym przypadku została użyta heurystyka, bo dla zwykłego komputeraz obliczenie poprawnej najkrótszej drogi zajmuje n! czasu, dla kilku miast komputer szybko znajdzie najlepszą trasę, ale dla kilkudziesięciu miast potrzebny czas wynosi więcej niż czas istnienie wszechświata. W tym przypadku została użyta najprostrza heurystyka, która nie jest optymalna.


### [4.1] Małe miasto

Tak prezentują się wyniki heurystyki, w której to algortym szuka najkrótszej drogi dla grafu małego miasta:

In [ ]:
start_time = time.perf_counter()
start_city = "21"
road, distance = distance_of_travelling_salesman(small_pathways, start_city)
end_time = time.perf_counter()
total_time = end_time - start_time
print(f"Cities that were visited from {start_city} are: ")
print(road)
print(f"and the distance is {distance}")
print(f"It took programme {total_time} for solving travelling salesman problem")

Cities that were visited from 21 are: 
['21', '39', '1', '34', '25', '42', '11', '4', '33', '44', '13', '6', '22', '23', '47', '29', '5', '8', '45', '28', '17', '40', '24', '37', '46', '38', '43', '48', '30', '19', '14', '18', '10', '15', '32', '2', '27', '7', '31', '49', '12', '36', '21']
and the distance is 631.46
It took programme 0.040660290978848934 for solving travelling salesman problem


Dla grafu o niewielkiej liczbie wierzchołków
zaimplementowana heurystyka działa błyskawicznie, zwracając
wynik w zaledwie około 0.04 sekundy. Algorytm prawidłowo
wyznaczył cykl komiwojażera. Co najważniejsze, dzięki 
wykorzystaniu Dijkstry pod spodem, z sukcesem ominął problem
"ślepych zaułków" i odizolowanych części grafu, znajdując
legalną trasę omijającą luki w strukturze miejskiej.

### [4.2] Średnie Miasto

In [ ]:
start_time = time.perf_counter()
start_city = "70"
road, distance = distance_of_travelling_salesman(mid_pathways, start_city)
end_time = time.perf_counter()
total_time = end_time - start_time
print(f"Cities that were visited from {start_city} are: ")
print(road)
print(f"and the distance is {distance}")
print(f"It took programme {total_time} for solving travelling salesman problem")

Cities that were visited from 70 are: 
['70', '8', '129', '61', '120', '56', '110', '102', '36', '39', '198', '62', '83', '131', '165', '65', '46', '148', '132', '119', '63', '99', '78', '125', '14', '12', '10', '95', '34', '85', '124', '75', '156', '117', '123', '19', '73', '88', '135', '106', '101', '25', '37', '186', '184', '55', '193', '9', '185', '92', '81', '109', '121', '126', '27', '49', '71', '7', '67', '58', '42', '199', '80', '190', '160', '192', '69', '38', '87', '33', '166', '162', '151', '97', '182', '64', '104', '174', '3', '179', '18', '136', '159', '53', '32', '21', '51', '66', '112', '82', '4', '158', '45', '171', '133', '175', '74', '168', '94', '79', '134', '54', '59', '20', '76', '157', '0', '114', '57', '30', '137', '191', '188', '161', '194', '89', '15', '72', '128', '180', '170', '149', '183', '5', '100', '167', '130', '178', '146', '196', '16', '43', '111', '107', '103', '155', '187', '98', '50', '172', '195', '90', '93', '164', '173', '28', '96', '181', '105',

### [4.3] Duże miasto

In [ ]:
# start_time = time.perf_counter()
# start_city = "500"
# road, distance = distance_of_travelling_salesman(big_pathways, start_city)
# end_time = time.perf_counter()
# total_time = end_time - start_time
# print(f"Cities that were visited from {start_city} are: ")
# print(road)
# print(f"and the distance is {distance}")
# print(f"It took programme {total_time} for solving travelling salesman problem")

Eksperyment dla dużego grafu (1000 węzłów, ponad 20 000 krawędzi) 
zakończył się klasyczną "eksplozją kombinatoryczną". O ile prosta heurystyka 
z pod-wywołaniami Dijkstry świetnie sprawdziła się dla 200 miast, tak przy 
grafie rzędu N=1000 ilość wymaganych operacji rośnie do miliardów. 
Każdy krok komiwojażera wymusza weryfikację setek nieodwiedzonych sąsiadów, 
z których każdy odpala pełną Dijkstrę na potężnej siatce 20 tysięcy dróg.

## [5] Wnioski

Przeprowadzone badania na trójstopniowym modelu sieci miejskiej pozwoliły na sformułowanie kluczowych wniosków.

- Wydajność struktur danych: Oparcie projektu o autorską słownikową reprezentację grafu okazało się bardzo trafną decyzją. Pozwoliło to na błyskawiczny dostęp do sąsiadów i wyeliminowało spowolnienia w krytycznych fazach działania programu.

- Skalowalność algorytmu Dijkstry: Zastosowanie kopca do obsługi kolejki priorytetowej zapewniło rewelacyjną szybkość obliczeń. Szukanie najkrótszej ścieżki oraz modyfikacja uwzględniająca dynamiczne godziny przejazdów sprawdziły się doskonale nawet w najbardziej zagęszczonej siatce dróg.

- Specyfika gęstych grafów: Obliczone czasy dojazdów w największym mieście potwierdzają teorię małego świata. Duża liczba skrzyżowań pozwala algorytmowi optymalizować trasę do zaledwie kilku kluczowych skoków między głównymi arteriami.

- Bariera problemów NP-trudnych: Implementacja problemu komiwojażera obnażyła fizyczne limity sprzętowe. Zaproponowana heurystyka z łatwością poradziła sobie z omijaniem ślepych zaułków na mniejszych mapach, jednak zderzenie z potężną liczbą krawędzi dużego miasta doprowadziło do ostatecznej eksplozji kombinatorycznej.

-Konieczność stosowania metaheurystyk: Eksperyment udowodnił, że klasyczne podejścia zachłanne są absolutnie niewystarczające dla ogromnych systemów. Rozwiązanie tego problemu w tak złożonym środowisku wymagałoby wdrożenia znacznie bardziej zaawansowanych algorytmów przybliżonych.